# Projek PDS - Rekomendasi Lokasi UMKM
## Notebook 01: Eksplorasi Awal & Ekstraksi Data OpenStreetMap (OSM)

Notebook ini bertujuan untuk melakukan eksplorasi data (*Exploratory Data Analysis - EDA*) awal terhadap data Point of Interest (POI) UMKM dari OpenStreetMap (OSM) menggunakan **Overpass API**.

### Tujuan Analisis:
1. Memahami struktur respon JSON dari Overpass API (nodes & ways).
2. Menelaah kelengkapan atribut (nama, tag amenity, shop, craft, alamat, dll).
3. Memetakan sebaran spasial UMKM di wilayah studi secara interaktif.
4. Mengidentifikasi distribusi kategori/sektor usaha untuk bahan pemodelan rekomendasi lokasi.

### 1. Import Library & Persiapan Lingkungan

In [ ]:
import sys
import json
import requests
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# Setting styling grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

# Tambahkan direktori root projek ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

print(f"Project Root: {ROOT_DIR}")

### 2. Konfigurasi Area Studi & Query Overpass API

Sebagai sampel eksplorasi, kita menggunakan wilayah **Kota Bandung** (atau area perkotaan yang dipilih). Bounding box didefinisikan dengan format `(min_lat, min_lon, max_lat, max_lon)`.

In [ ]:
# Import konfigurasi dari scripts
try:
    from scripts.dataset1_osm.config import BBOX_WILAYAH, OVERPASS_ENDPOINTS, SEKTOR_MAPPING
    wilayah_info = BBOX_WILAYAH.get("bandung_kota")
    bbox = wilayah_info["bbox"]
    print(f"Wilayah: {wilayah_info['nama']}")
    print(f"BBox: {bbox}")
except ImportError:
    # Fallback jika dijalankan terpisah
    bbox = (-6.9300, 107.5900, -6.8900, 107.6400) # Pusat Kota Bandung (sampel lebih kecil untuk EDA cepat)
    print(f"Menggunakan fallback BBox sampel: {bbox}")

### 3. Eksekusi Query Contoh Overpass API
Mari kita coba menarik POI kategori `amenity` (seperti cafe, restaurant, fast_food) dan `shop` (convenience, supermarket, bakery, dll).

In [ ]:
s, w, n, e = bbox
overpass_url = "https://overpass-api.de/api/interpreter"

# Query sampel Overpass QL
sample_query = f"""[out:json][timeout:60];
(
  node["amenity"~"restaurant|cafe|fast_food|marketplace"]({s},{w},{n},{e});
  node["shop"~"convenience|supermarket|bakery|clothes"]({s},{w},{n},{e});
);
out body;
"""

print("Mengirim query ke Overpass API...")
response = requests.post(overpass_url, data={'data': sample_query}, timeout=75)
print(f"Status Code: {response.status_code}")

data_json = response.json()
elements = data_json.get('elements', [])
print(f"Total POI ditemukan: {len(elements)}")

### 4. Ekstraksi dan Tabulasi Data ke Pandas DataFrame

In [ ]:
parsed_data = []
for el in elements:
    tags = el.get('tags', {})
    parsed_data.append({
        'osm_id': el.get('id'),
        'latitude': el.get('lat'),
        'longitude': el.get('lon'),
        'nama': tags.get('name', 'Tanpa Nama'),
        'amenity': tags.get('amenity'),
        'shop': tags.get('shop'),
        'street': tags.get('addr:street'),
        'opening_hours': tags.get('opening_hours')
    })

df_sample = pd.DataFrame(parsed_data)
df_sample.head(10)

### 5. Analisis Kelengkapan & Distribusi Kategori

In [ ]:
# Cek persentase nama yang terisi
pct_named = (df_sample['nama'] != 'Tanpa Nama').mean() * 100
print(f"Persentase POI dengan Nama Terisi: {pct_named:.2f}%")

# Gabungkan kategori amenity dan shop untuk melihat jenis usaha terbanyak
df_sample['kategori'] = df_sample['amenity'].fillna(df_sample['shop'])

plt.figure(figsize=(10, 5))
df_sample['kategori'].value_counts().head(10).plot(kind='barh', color='#3498db')
plt.title('Top 10 Kategori Usaha Terbanyak (Sampel OSM)')
plt.xlabel('Jumlah Titik')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 6. Visualisasi Spasial Interaktif dengan Folium

In [ ]:
if not df_sample.empty:
    # Titik tengah peta
    center_lat = df_sample['latitude'].mean()
    center_lon = df_sample['longitude'].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

    for _, row in df_sample.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4,
            popup=f"{row['nama']} ({row['kategori']})",
            tooltip=row['nama'],
            color='#e74c3c' if row['amenity'] else '#2ecc71',
            fill=True,
            fill_opacity=0.7
        ).add_to(m)

    # Tampilkan peta di notebook
    display(m)
else:
    print("Dataframe kosong, tidak dapat membuat peta.")

### 7. Catatan Temuan & Langkah Selanjutnya
- **Kerapatan Data**: Data OSM di pusat kota memiliki densitas yang cukup padat untuk sektor F&B (kafe & restoran) dan ritel (convenience store).
- **Tantangan**: Sebagian titik belum memiliki atribut nama spesifik (`Tanpa Nama`) atau atribut jam operasional yang lengkap.
- **Rencana Pipeline Otomatis**: Gunakan script modul `scripts/dataset1_osm/main.py` untuk mengekstraksi seluruh batas administratif kota dengan grid sampling dan deduplikasi otomatis.